# 07 · Guide depletion

A guide whose target is essential kills the cells that receive it, so the guide
is under-represented in the screen relative to the plasmid pool it was
delivered in. This notebook compares the two proportions per target gene.

The result is diagnostic: it does not filter anything. It is reported in the
supplementary figures and is worth reading before interpreting a knockout whose
cell count is low.

**Reads** `par_save_filename_5` and `par_initial_guide_pool_file`.
**Writes** `par_guide_depletion_file`.

`par_initial_guide_pool_file` has one row per guide and the number of cells
that guide contributed to the pool.

## Setup

In [1]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.stats.multitest as smm

## Count cells per guide in the screen

Only cells carrying exactly one guide are counted, so that a cell is not
credited to two guides.

In [2]:
adata = sc.read(par_save_filename_5)
guide_names = list(adata.uns["feature_barcode_names"])

carried = (adata.obs[guide_names] > 0).astype(int)
single = carried[carried.sum(axis=1) == 1]
print(f"cells carrying exactly one guide: {single.shape[0]}")

screen_counts = single.sum(axis=0).rename("nCellsScreen").reset_index()
screen_counts.columns = ["Guide", "nCellsScreen"]
print(f"guides seen: {(screen_counts.nCellsScreen > 0).sum()} of {len(guide_names)}")

cells carrying exactly one guide: 341664
guides seen: 3716 of 3720


## Join the pool composition and aggregate to genes

Guide names in the pool table use `-` where the screen uses `_`; gene names
that legitimately contain a hyphen have to survive that substitution, so they
are restored explicitly.

In [3]:
pool = pd.read_csv(par_initial_guide_pool_file)
pool.columns = ["Guide", "nCellsPool"]
pool["nCellsPool"] = pool["nCellsPool"].astype(int)
pool["Guide"] = pool["Guide"].replace("-", "_", regex=True)

# gene names that really do contain a hyphen
for name in ["Rnf8-cmtr1", "Siah1-ps1", "Siah1-ps2"]:
    pool["Guide"] = pool["Guide"].str.replace(name.replace("-", "_"), name, regex=False)

# Keep only rows that name a guide the screen actually carries. The pool table
# is a summary export and can hold rows that are not guides; anything that does
# not correspond to a guide is not comparable and is reported rather than
# silently carried into the totals.
known = set(screen_counts.Guide)
not_guides = sorted(set(pool.Guide) - known)
pool = pool[pool.Guide.isin(known)].copy()

print(f"pool rows: {len(pool)} usable, {len(not_guides)} not guides in the screen")
if not_guides:
    print(f"  dropped: {', '.join(not_guides[:10])}{' ...' if len(not_guides) > 10 else ''}")

missing_from_pool = sorted(known - set(pool.Guide))
if missing_from_pool:
    print(f"guides in the screen with no pool entry: {len(missing_from_pool)}")

result = pd.merge(pool, screen_counts, on="Guide", how="inner", validate="one_to_one")
result["targetGene"] = ["_".join(g.split("_")[:-1]) for g in result.Guide]

per_gene = result.groupby("targetGene")[["nCellsPool", "nCellsScreen"]].sum()
per_gene = per_gene.drop(index=[par_not_target_control_prefix.rstrip("_"),
                                par_nongene_site_control_prefix.rstrip("_")],
                          errors="ignore")
print(f"target genes: {per_gene.shape[0]}")

pool rows: 3719 usable, 1 not guides in the screen
  dropped: Fullstats
guides in the screen with no pool entry: 1
target genes: 1130


## Test each gene for depletion

A one-sided two-proportion z-test: is the gene's share of screen cells smaller
than its share of pool cells?

In [4]:
total_screen = per_gene.nCellsScreen.sum()
total_pool = per_gene.nCellsPool.sum()

per_gene["nCellsPoolPerc"] = per_gene.nCellsPool / total_pool
per_gene["nCellsScreenPerc"] = per_gene.nCellsScreen / total_screen
per_gene["nCellsScreenPoolPercDif"] = per_gene.nCellsScreenPerc - per_gene.nCellsPoolPerc

# Both directions, as in the original analysis: "larger" asks whether the gene's
# share of the pool exceeds its share of the screen (depleted from the screen),
# "smaller" asks the reverse (enriched in it).
stat_larger, pval_larger, stat_smaller, pval_smaller = [], [], [], []

for gene, row in per_gene.iterrows():
    count = np.array([row.nCellsPool, row.nCellsScreen])
    nobs = np.array([total_pool, total_screen])

    s1, p1 = proportions_ztest(count, nobs, alternative="larger")
    s2, p2 = proportions_ztest(count, nobs, alternative="smaller")
    stat_larger.append(s1); pval_larger.append(p1)
    stat_smaller.append(s2); pval_smaller.append(p2)

per_gene["statLarger"] = stat_larger
per_gene["pValLarger"] = pval_larger
per_gene["statSmaller"] = stat_smaller
per_gene["pValSmaller"] = pval_smaller

per_gene["FDR_larger"] = smm.multipletests(per_gene.pValLarger, method="fdr_bh")[1]
per_gene["FDR_smaller"] = smm.multipletests(per_gene.pValSmaller, method="fdr_bh")[1]

per_gene = per_gene.sort_values("FDR_larger")

print(f"depleted from the screen at FDR < 0.1: {(per_gene.FDR_larger < 0.1).sum()}")
print(f"enriched in the screen  at FDR < 0.1: {(per_gene.FDR_smaller < 0.1).sum()}")
print()
print(per_gene.head(15).to_string())

depleted from the screen at FDR < 0.1: 419
enriched in the screen  at FDR < 0.1: 348

               nCellsPool  nCellsScreen  nCellsPoolPerc  nCellsScreenPerc  nCellsScreenPoolPercDif  statLarger    pValLarger  statSmaller  pValSmaller    FDR_larger  FDR_smaller
targetGene                                                                                                                                                                       
Mdm2                 1880            30        0.001124          0.000101                -0.001023   16.523617  1.240316e-61    16.523617          1.0  1.401557e-58          1.0
Copa                 1896            72        0.001133          0.000242                -0.000891   14.184884  5.682656e-46    14.184884          1.0  3.210701e-43          1.0
Gnb4                 1356            26        0.000811          0.000087                -0.000723   13.730857  3.317202e-43    13.730857          1.0  1.249480e-40          1.0
Traip                185

## Write

In [5]:
Path(par_guide_depletion_file).parent.mkdir(parents=True, exist_ok=True)
per_gene.to_csv(par_guide_depletion_file)
print(f"written: {par_guide_depletion_file}")

written: TextFiles/NoOfCellsPerGuide_GeneLevel.csv
